In [1]:
import numpy as np
import pandas as pd
import cudaq
import sys
import os
import shutil
import faulthandler
from scipy.linalg import expm
from math import sqrt
from tqdm import tqdm
import torch
from typing import List, Tuple
sys.path.append(os.path.abspath(".."))
from Utils.qaoaCUDAQ import po_normalize, ret_cov_to_QUBO, qubo_to_ising, process_ansatz_values, kernel_flipped, all_state_to_return, find_budget

In [2]:
e = 0
N_ASSETS = 3
TARGET_QUBIT_IN = 2
q = 1.5
lamb = 0.05

device = torch.device("cuda")
DUPLICATE_ASSET = False
min_P, max_P = 108, 216

In [3]:
data_cov_pd = pd.read_csv("../dataset/top_50_us_stocks_data_20250526_011226_covariance.csv")
data_ret_p_pd = pd.read_csv("../dataset/top_50_us_stocks_returns_price.csv")

data_ret_p_pd = data_ret_p_pd[(data_ret_p_pd["Price"] > min_P) & (data_ret_p_pd["Price"] < max_P)]
data_cov_pd = data_cov_pd.loc[data_cov_pd["Ticker"].isin(data_ret_p_pd["Ticker"])].reset_index(drop=True)
data_cov_pd = data_cov_pd[["Ticker"] + data_cov_pd["Ticker"].tolist()]

In [4]:
np.random.seed(911 + 991 * e + 997 * N_ASSETS)
state = np.random.get_state()
# asset_idx = np.random.choice(data_cov_pd.shape[0], max(TARGET_ASSET), replace=False)
asset_idx = np.random.choice(data_cov_pd.shape[0], N_ASSETS, replace=DUPLICATE_ASSET)
# print(asset_idx)
# asset_idx = np.array([0, 18, 27, 32, 41])
# data_cov = data_cov_pd.drop("Ticker", axis=1)
data_cov = data_cov_pd.drop("Ticker", axis=1).to_numpy()[asset_idx, :][:, asset_idx]
stock_names = data_ret_p_pd["Company_Name"].to_numpy()[asset_idx]
# print("Selected Stocks: ", stock_names)
data_ret_p = data_ret_p_pd.drop("Ticker", axis=1)
# print(data_ret_p.index[asset_idx].to_numpy())
asset_idx_raw = data_ret_p.index[asset_idx].to_numpy()
data_ret_p = data_ret_p.drop("Company_Name", axis=1).to_numpy()[asset_idx, :]

data_ret = data_ret_p[:, 0]
data_p = data_ret_p[:, 1]
print("Selected Stocks: ", stock_names)
print("Selected Stocks Price: ", data_p)
print("Selected Stocks Return: ", data_ret)

Selected Stocks:  ['Philip Morris International Inc.' 'Abbott Laboratories' 'Broadcom Inc.']
Selected Stocks Price:  [158.72999573 132.0383606  167.42999268]
Selected Stocks Return:  [0.00060737 0.00060593 0.00142008]


In [5]:
np.random.set_state(state)
weighted = np.random.uniform(0, 1)
B_mi, B_ma = find_budget(TARGET_QUBIT_IN * N_ASSETS, data_p, min_P, max_P, min_mix_mode=True)
B = B_mi * weighted + B_ma * (1 - weighted)

P = data_p[:N_ASSETS]
ret = data_ret[:N_ASSETS]
cov = data_cov[:N_ASSETS, :N_ASSETS]
P_bb, ret_bb, cov_bb, n_qubit, n_max, C = po_normalize(B, P, ret, cov)
TARGET_QUBIT = n_qubit

# QUBOs of MAX PROBLEM
QU = ret_cov_to_QUBO(ret_bb, cov_bb, P_bb, lamb, q)
QU_lamb = ret_cov_to_QUBO(np.zeros_like(ret_bb), np.zeros_like(cov_bb), P_bb, lamb, 0.0)
QU_eval = ret_cov_to_QUBO(ret_bb, cov_bb, P_bb, 0.0, q)
QU_return = ret_cov_to_QUBO(ret_bb, np.zeros_like(cov_bb), np.zeros_like(P_bb), 0.0, 0.0)
QU_risk = ret_cov_to_QUBO(np.zeros_like(ret_bb), cov_bb, np.zeros_like(P_bb), 0.0, q)

# Hamiltonians of MIN PROBLEM
H_ansatz = -qubo_to_ising(QU, lamb).canonicalize()
H_lamb = -qubo_to_ising(QU_lamb, lamb).canonicalize()
H_eval = -qubo_to_ising(QU_eval, 0.0).canonicalize()
H_return = -qubo_to_ising(QU_return, 0.0).canonicalize()
H_risk = -qubo_to_ising(QU_risk, 0.0).canonicalize()

idx_1_use, coeff_1_use, idx_2_a_use, idx_2_b_use, coeff_2_use = process_ansatz_values(H_ansatz)

In [6]:
n_b = 1<<TARGET_QUBIT
H = H_ansatz.to_matrix()
print("Is Hermitian:", (H == np.conj(H).T).all())
eigvals, eigvecs = np.linalg.eigh(H)

eigvals = np.random.rand(n_b) * 3
eigvals = np.sort(eigvals)
# eigvals = eigvals - eigvals[0]
# eigvals[1:] += 1

H = eigvecs @ np.diag(eigvals) @ eigvecs.T
# print("Modified H:\n", H)

for i in range(len(eigvals)):
    print(f"Eigenvalue {i}: {eigvals[i]}")
    print(f"Eigenvector {i}:\n{eigvecs[:, i]}\n")

ground_state = eigvecs[:, 0]
print("Ground state:\n", ground_state)

# print(np.linalg.eigh(H))
# print(np.linalg.eig(H))

# b = np.random.rand(n_b)
b = np.ones(n_b)
b = b / np.linalg.norm(b)
print("Vector b:\n", b)

spectral_gap = eigvals[1] - eigvals[0]
spectral_radius = max(abs(eigvals))
print("\nSpectral gap:", spectral_gap)
print("Spectral radius:", spectral_radius)

ss = spectral_gap / (12 * spectral_radius**3)
F0 = abs(b @ ground_state)**2
print("F0:", F0)
q = 1 - ss * F0 * spectral_gap
print("q:", q)
print("Recommended step size (s):", ss)
assert spectral_radius > 1

Is Hermitian: True
Eigenvalue 0: 0.026456695199172486
Eigenvector 0:
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j]

Eigenvalue 1: 0.0762567911048947
Eigenvector 1:
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j


In [7]:
def commutator(A, B):
    return A @ B - B @ A

In [8]:
assert False

AssertionError: 

In [42]:
# tau = 200
# tau = 0.2
N = 60000
tau = ss * N
# N = 100

print("tau:", tau)


# e^(-tau H)
exp_H = expm(-tau * H)
print("Exponential of -tau * H:\n", exp_H)

result = exp_H @ b
result = result / np.linalg.norm(result)
print("Fidelity with ground state:", abs(result @ ground_state)**2)

print("Result of normed exp(-tau * H) @ b:", result)
# print("Result of normed exp(-tau * H) @ b:", np.round(result))

tau: 9.540502251801792
Exponential of -tau * H:
 [[2.04823043e-10+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j ...
  0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j]
 [0.00000000e+00+0.j 2.46788597e-07+0.j 0.00000000e+00+0.j ...
  0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j]
 [0.00000000e+00+0.j 0.00000000e+00+0.j 7.62544591e-04+0.j ...
  0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j]
 ...
 [0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j ...
  2.67437748e-11+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j]
 [0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j ...
  0.00000000e+00+0.j 1.40157521e-12+0.j 0.00000000e+00+0.j]
 [0.00000000e+00+0.j 0.00000000e+00+0.j 0.00000000e+00+0.j ...
  0.00000000e+00+0.j 0.00000000e+00+0.j 5.12416601e-13+0.j]]
Fidelity with ground state: 0.577363176706021
Result of normed exp(-tau * H) @ b: [2.00319715e-10+0.j 2.41362597e-07+0.j 7.45778957e-04+0.j
 1.15929260e-01+0.j 8.14812456e-08+0.j 4.36132701e-05+0.j
 1.083943

In [44]:
s = tau / N

s = 1
N = 5

bb = b.copy()
bb_t = torch.tensor(bb, device=device, dtype=torch.complex64)
H_t = torch.tensor(H, device=device, dtype=torch.complex64)
for i in tqdm(range(N)):
    densi = torch.outer(bb_t, bb_t)
    Q = torch.matrix_exp(s * (densi @ H_t - H_t @ densi))
    bb_t = Q @ bb_t
bb = bb_t.cpu().numpy()
result = bb
print("Fidelity with ground state:", abs(result @ ground_state)**2)
print("Result:", result)
print("Norm:", np.linalg.norm(result))
print(np.round(result))

100%|██████████| 5/5 [00:00<00:00, 2367.26it/s]

Fidelity with ground state: 0.1629412507318193
Result: [-1.26098692e-02+0.j  6.74561888e-05+0.j  1.12136779e-02+0.j
  1.92684084e-01+0.j  3.07826675e-04+0.j  5.60870278e-04+0.j
  1.87243804e-01+0.j  3.10794506e-02+0.j  4.52290260e-04+0.j
  1.20496884e-01+0.j  7.27498382e-02+0.j  2.47457137e-07+0.j
  1.15113378e-01+0.j  9.37137380e-02+0.j  3.43660504e-05+0.j
  7.89709797e-04+0.j  3.99313067e-05+0.j  2.20974870e-02+0.j
  2.79042542e-01+0.j  5.21724392e-03+0.j  3.48270033e-03+0.j
  2.53743201e-01+0.j  2.32777391e-02+0.j  4.79796654e-06+0.j
  1.26534685e-01+0.j  6.47479147e-02+0.j  1.92492735e-07+0.j
  1.09289680e-03+0.j  7.88682178e-02+0.j  2.47312128e-05+0.j
  8.79001920e-04+0.j -4.26650643e-02+0.j  2.39245947e-02+0.j
  3.38340729e-01+0.j  1.18592975e-03+0.j  2.17797351e-04+0.j
  4.03659821e-01+0.j  1.56580247e-02+0.j  6.48450223e-05+0.j
 -8.19899794e-03+0.j  3.22883092e-02+0.j -9.93144567e-07+0.j
  7.82930118e-04+0.j -7.39918053e-02+0.j -4.10900896e-08+0.j
  9.95175564e-04+0.j -4.400248

In [45]:
s = tau / N

s = 1
N = 5

bb = b.copy()
# for i in range(N):
#     densi = np.outer(bb, bb)
#     E1 = expm(-1j*sqrt(s)*H)
#     E2 = expm(1j*sqrt(s)*densi)
#     E3 = expm(1j*sqrt(s)*H)
#     bb = np.exp(-1j*sqrt(s)) * E3 @ E2 @ E1 @ bb
#     print(f"iter {i+1}")
#     F_k = abs(bb @ ground_state)**2
#     print(f"F_{i+1}:", F_k)
#     print("bb:", bb)
#     lower_bound = 1 - q**(i)
#     print(f"Lower bound for F_{i+1}:", lower_bound)
#     print()
bb_t = torch.tensor(bb, device=device, dtype=torch.complex64)
H_t = torch.tensor(H, device=device, dtype=torch.complex64)
ground_state_t = torch.tensor(ground_state, device=device, dtype=torch.complex64)
E1 = torch.matrix_exp(-1j*sqrt(s)*H_t)
E3 = torch.matrix_exp(1j*sqrt(s)*H_t)
E1_dagger = torch.adjoint(E1)
for i in tqdm(range(N)):
    densi = torch.outer(bb_t, bb_t)
    E2 = torch.matrix_exp(1j*sqrt(s)*densi)
    E2_dagger = torch.adjoint(E2)
    E321 = E3 @ E2 @ E1
    E321_dagger = torch.adjoint(E321)
    print(torch.trace(E2_dagger @ E2))
    eigvalss, eigvecss = torch.linalg.eig(E321)
    print("Eigenvalues of E321:", torch.sort(torch.abs(eigvalss)))
    bbb = bb_t.clone()
    bb_t = E321 @ bb_t
    if (i+1) % 1 == 0:
        F_k = abs(bb_t @ ground_state_t)**2
        print(f"iter {i+1}")
        print("norm of bb_t:", torch.norm(bb_t).item())
        print(f"F_{i+1}:", F_k)
        lower_bound = 1 - q**(i+1)
        print(f"Lower bound for F_{i+1}:", lower_bound)
        print()
bb = bb_t.cpu().numpy()

print("Fidelity with ground state:", abs(bb @ ground_state)**2)
print("Result of DB-QITE:", bb)
print("Abs of DB-QITE result:", np.abs(bb) * np.sign(bb.real))
print("Norm of DB-QITE result:", np.linalg.norm(bb))

100%|██████████| 5/5 [00:00<00:00, 57.23it/s]

tensor(64.0000+7.9007e-10j, device='cuda:0')
Eigenvalues of E321: torch.return_types.sort(
values=tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], device='cuda:0'),
indices=tensor([ 2,  4, 37,  8,  0, 18, 29, 45, 47,  1, 19, 43, 63,  5, 13, 15, 27, 28,
        33, 41, 42, 62, 11, 12, 20, 25, 51, 53, 54, 60, 58, 48, 59, 10, 38, 57,
        24, 39, 44, 56,  9, 31, 34, 36, 61, 14, 26, 32, 52, 17, 21, 40, 46, 49,
        55,  3, 50,  6, 16, 23, 30, 35,  7, 22], device='cu

In [ ]:
print(torch.trace(E1 @ E3))
print(np.exp(-1j*sqrt(s)))

tensor(64.0000-2.4615e-08j, device='cuda:0')
(0.5403023058681398-0.8414709848078965j)


In [ ]:
print(s)

1


In [ ]:
# print(densi)
densi = torch.outer(bbb, bbb)
dd = torch.matrix_exp(1j*10*densi)
dd_dagger = torch.adjoint(dd)
print(torch.trace(dd_dagger @ dd))
print(torch.diag(dd_dagger @ dd))

tensor(63.1706+7.7922e-10j, device='cuda:0')
tensor([0.9989-8.5414e-12j, 0.9935-1.1237e-11j, 0.9770-7.1038e-11j,
        0.9683-2.4157e-11j, 0.9945+8.0748e-11j, 0.9842+2.9279e-11j,
        0.9683+2.2821e-10j, 0.9735+5.4281e-11j, 0.9845+1.8273e-10j,
        0.9691+8.2077e-11j, 0.9706+1.4291e-11j, 0.9901+1.8677e-10j,
        0.9692-1.8612e-11j, 0.9698+8.3285e-11j, 0.9872-1.4682e-10j,
        0.9955-2.3002e-12j, 0.9932-2.5116e-11j, 0.9747-1.1287e-10j,
        0.9681+2.4944e-10j, 0.9793+1.5898e-10j, 0.9804+1.4557e-10j,
        0.9681-3.8747e-10j, 0.9745+7.7563e-11j, 0.9925-3.1026e-12j,
        0.9690-2.1435e-10j, 0.9709+3.6719e-11j, 0.9903+2.0131e-10j,
        0.9962+1.4363e-11j, 0.9703-4.1826e-11j, 0.9875+8.1231e-11j,
        0.9956+7.0181e-12j, 1.0012-1.1725e-11j, 0.9744-7.8004e-11j,
        0.9683+7.5205e-11j, 0.9828-1.6147e-10j, 0.9943-3.7672e-11j,
        0.9686+1.5318e-10j, 0.9759+5.4422e-11j, 0.9935+6.6049e-11j,
        0.9984+5.9184e-11j, 0.9734+8.2236e-12j, 0.9909-8.7794e-11j,
   

In [ ]:
eig_valss, eig_vecss = torch.linalg.eig(dd)
print(abs(eig_valss))
print(eig_vecss[:, 1])

tensor([1.0000e+00, 6.0008e-04, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00,
        1.0000e+00, 1.0000e+00, 1.0000e+00, 1.0000e+00], device='cuda:0')
tensor([0.0454-0.0074j, 0.0842+0.0355j, 0.1493+0.0382j, 0.1826+0.0132j,
        0.0764+0.0310j, 0.1263+0.0436j, 0.1822+0.0137j

In [ ]:
print(bbb)

tensor([0.0456+0.0059j, 0.0706+0.0581j, 0.1321+0.0793j, 0.1712+0.0649j,
        0.0643+0.0515j, 0.1085+0.0779j, 0.1707+0.0652j, 0.1442+0.0773j,
        0.1074+0.0777j, 0.1637+0.0696j, 0.1561+0.0733j, 0.0867+0.0698j,
        0.1630+0.0700j, 0.1599+0.0716j, 0.0977+0.0748j, 0.0581+0.0432j,
        0.0721+0.0595j, 0.1399+0.0782j, 0.1771+0.0602j, 0.1245+0.0796j,
        0.1210+0.0795j, 0.1756+0.0615j, 0.1406+0.0781j, 0.0759+0.0627j,
        0.1645+0.0691j, 0.1544+0.0740j, 0.0858+0.0693j, 0.0533+0.0346j,
        0.1573+0.0728j, 0.0969+0.0745j, 0.0571+0.0416j, 0.0458-0.0079j,
        0.1409+0.0780j, 0.1802+0.0574j, 0.1130+0.0787j, 0.0660+0.0535j,
        0.1830+0.0546j, 0.1358+0.0789j, 0.0707+0.0582j, 0.0460+0.0097j,
        0.1447+0.0771j, 0.0830+0.0676j, 0.0507+0.0286j, 0.0469-0.0157j,
        0.0888+0.0709j, 0.0557+0.0392j, 0.0458-0.0083j, 0.0528-0.0337j,
        0.1743+0.0625j, 0.1091+0.0780j, 0.0641+0.0513j, 0.0457-0.0071j,
        0.1334+0.0792j, 0.0680+0.0556j, 0.0454-0.0009j, 0.0497-0

In [9]:
k = 5
U = [None for _ in range(k+1)]
def conj(l):
    ll = [(a, b, -c) for (a, b, c) in l[::-1]]
    return ll

def dfs(n, i):
    if i == 0:
        return []
    if U[i] is not None:
        return U[i]
    U[i] = dfs(n, i-1) + [("H", i-1, -1)] + conj(dfs(n, i-1)) + [("D", i-1, 1)] + dfs(n, i-1) + [("H", i-1, 1)]
    return U[i]

In [10]:
dfs(k, k)
for i in range(k+1):
    print(f"U_{i}:", U[i])
# a = [('H', 0, 1), ('D', 0, -1)]
# print(conj(a))
type_l, tim_l, abs_l = zip(*U[k])
type_l = [0 if t == "H" else 1 for t in type_l]
tim_l = np.array(tim_l)
abs_l = np.array(abs_l)
type_l = np.array(type_l)

print("tim_l:", tim_l)
print("abs_l:", abs_l)
print("type_l:", type_l)

U_0: None
U_1: [('H', 0, -1), ('D', 0, 1), ('H', 0, 1)]
U_2: [('H', 0, -1), ('D', 0, 1), ('H', 0, 1), ('H', 1, -1), ('H', 0, -1), ('D', 0, -1), ('H', 0, 1), ('D', 1, 1), ('H', 0, -1), ('D', 0, 1), ('H', 0, 1), ('H', 1, 1)]
U_3: [('H', 0, -1), ('D', 0, 1), ('H', 0, 1), ('H', 1, -1), ('H', 0, -1), ('D', 0, -1), ('H', 0, 1), ('D', 1, 1), ('H', 0, -1), ('D', 0, 1), ('H', 0, 1), ('H', 1, 1), ('H', 2, -1), ('H', 1, -1), ('H', 0, -1), ('D', 0, -1), ('H', 0, 1), ('D', 1, -1), ('H', 0, -1), ('D', 0, 1), ('H', 0, 1), ('H', 1, 1), ('H', 0, -1), ('D', 0, -1), ('H', 0, 1), ('D', 2, 1), ('H', 0, -1), ('D', 0, 1), ('H', 0, 1), ('H', 1, -1), ('H', 0, -1), ('D', 0, -1), ('H', 0, 1), ('D', 1, 1), ('H', 0, -1), ('D', 0, 1), ('H', 0, 1), ('H', 1, 1), ('H', 2, 1)]
U_4: [('H', 0, -1), ('D', 0, 1), ('H', 0, 1), ('H', 1, -1), ('H', 0, -1), ('D', 0, -1), ('H', 0, 1), ('D', 1, 1), ('H', 0, -1), ('D', 0, 1), ('H', 0, 1), ('H', 1, 1), ('H', 2, -1), ('H', 1, -1), ('H', 0, -1), ('D', 0, -1), ('H', 0, 1), ('D', 1, -

In [ ]:
@cudaq.kernel
def kernel_DB_QITE(step_size: List[float], qubit_count: int, idx_1: List[int], coeff_1: List[float], idx_2_a: List[int], idx_2_b: List[int], coeff_2: List[float], type_l: List[int], tim_l: List[int], abs_l: List[float]):
    qreg = cudaq.qvector(qubit_count)
    h(qreg)

    for i in range(len(type_l)):
        # H
        if type_l[i] == 0:
            for j in range(len(idx_1)):
                rz(2 * abs_l[i] * coeff_1[j] * step_size[tim_l[i]], qreg[idx_1[j]])
            for j in range(len(idx_2_a)):
                x.ctrl(qreg[idx_2_a[j]], qreg[idx_2_b[j]])
                rz(2  * abs_l[i] * coeff_2[j] * step_size[tim_l[i]], qreg[idx_2_b[j]])
                x.ctrl(qreg[idx_2_a[j]], qreg[idx_2_b[j]])
        
        # D
        else:
            # h(qreg)
            # rz(-1 * abs_l[i] * step_size[tim_l[i]], qreg[qubit_count-1])
            # x.ctrl(qreg[0:qubit_count-1], qreg[qubit_count-1])
            # rz(-1 * abs_l[i] * step_size[tim_l[i]], qreg[qubit_count-1])
            # x.ctrl(qreg[0:qubit_count-1], qreg[qubit_count-1])
            # rz(2 * abs_l[i] * step_size[tim_l[i]], qreg[qubit_count-1])
            # h(qreg)

            ### I'M SORRY DEAR, BUT THIS IS NOT e^(i|u><u|) YOU'RE LOOKING FOR
            h(qreg)
            x(qreg)
            rz.ctrl(1 * abs_l[i] * step_size[tim_l[i]], qreg[0:qubit_count-1], qreg[qubit_count-1])
            x(qreg)
            h(qreg)

In [69]:
print(cudaq.draw(kernel_DB_QITE, [1.0]*k, TARGET_QUBIT, idx_1_use, coeff_1_use, idx_2_a_use, idx_2_b_use, coeff_2_use, type_l, tim_l, abs_l))

     ╭───╮╭─────────────╮                                                     »
q0 : ┤ h ├┤ rz(0.01506) ├──●────────────────────●────●─────────────────────●──»
     ├───┤├─────────────┤╭─┴─╮╭──────────────╮╭─┴─╮  │                     │  »
q1 : ┤ h ├┤ rz(0.03012) ├┤ x ├┤ rz(-0.01129) ├┤ x ├──┼─────────────────────┼──»
     ├───┤├─────────────┤╰───╯╰──────────────╯╰───╯╭─┴─╮╭───────────────╮╭─┴─╮»
q2 : ┤ h ├┤ rz(0.01254) ├──────────────────────────┤ x ├┤ rz(-0.004676) ├┤ x ├»
     ├───┤├─────────────┤                          ╰───╯╰───────────────╯╰───╯»
q3 : ┤ h ├┤ rz(0.02507) ├─────────────────────────────────────────────────────»
     ├───┤├─────────────┤                                                     »
q4 : ┤ h ├┤ rz(0.01582) ├─────────────────────────────────────────────────────»
     ├───┤├─────────────┤                                                     »
q5 : ┤ h ├┤ rz(0.03165) ├─────────────────────────────────────────────────────»
     ╰───╯╰─────────────╯               

In [66]:
result = cudaq.get_state(kernel_DB_QITE, [1]*k, TARGET_QUBIT, idx_1_use, coeff_1_use, idx_2_a_use, idx_2_b_use, coeff_2_use, type_l, tim_l, abs_l)
result_r = cudaq.get_state(kernel_flipped, result, TARGET_QUBIT)

In [67]:
# print(np.array(result_r))
uniform_state = np.ones_like(result) / np.sqrt(len(result))
print("Fidelity with ground state:", abs(uniform_state @ ground_state)**2)
print("Fidelity of result with ground state:", abs(result_r @ ground_state)**2)
print("Result of DB-QITE:", result)

Fidelity with ground state: 0.015625
Fidelity of result with ground state: 0.018571872414942906
Result of DB-QITE: SV: [(-0.096241,-0.0741407), (-0.101476,-0.0807223), (-0.104607,-0.0849335), (-0.105629,-0.0863547), (-0.100742,-0.0797658), (-0.104236,-0.0844232), (-0.105619,-0.0863408), (-0.104895,-0.0853306), (-0.10379,-0.0838127), (-0.105535,-0.0862226), (-0.105171,-0.0857163), (-0.1027,-0.0823418), (-0.105376,-0.0860001), (-0.105374,-0.0859982), (-0.103265,-0.0831008), (-0.0990439,-0.0775972), (-0.101743,-0.0810737), (-0.104768,-0.0851554), (-0.105682,-0.0864298), (-0.104489,-0.0847704), (-0.104405,-0.0846547), (-0.105681,-0.0864293), (-0.10485,-0.0852682), (-0.101908,-0.0812912), (-0.105605,-0.0863228), (-0.105136,-0.0856659), (-0.102556,-0.0821519), (-0.0978683,-0.0761295), (-0.105347,-0.0859609), (-0.103131,-0.0829201), (-0.0988037,-0.077294), (-0.0923998,-0.0696425), (-0.103816,-0.0757651), (-0.104684,-0.0768838), (-0.103285,-0.0750891), (-0.0996163,-0.0705664), (-0.104692,-0.07